In [ ]:
!pip install -q streamlit pyngrok opencv-python-headless pillow
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 37.4 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 3s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
from PIL import Image
import warnings

warnings.filterwarnings("ignore")

st.set_page_config(page_title="Stroke Region Extractor", layout="centered")

st.title("🧠 Brain Stroke Region Extraction + Severity")

# -----------------------------
# FILE UPLOAD
# -----------------------------
brain_file = st.file_uploader("Upload Brain CT Image", type=["jpg", "png", "jpeg"])

if brain_file:

    # Read image
    brain = np.array(Image.open(brain_file).convert("L"))

    st.subheader("📌 Input Image")
    st.image(brain, caption="Brain CT", use_container_width=True)

    # -----------------------------
    # PREPROCESSING
    # -----------------------------
    brain = cv2.GaussianBlur(brain, (3,3), 0)

    brain_pixels = brain[brain > 0]

    if len(brain_pixels) == 0:
        st.error("Invalid image!")
        st.stop()

    # -----------------------------
    # SEGMENTATION
    # -----------------------------
    mean = np.mean(brain_pixels)
    std = np.std(brain_pixels)
    threshold = int(mean + 1.0 * std)

    mask = cv2.inRange(brain, threshold, 255)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    pred_mask = np.zeros_like(mask)

    for cnt in contours:
        if cv2.contourArea(cnt) > 50:
            cv2.drawContours(pred_mask, [cnt], -1, 255, -1)

    # Smooth
    pred_mask = cv2.GaussianBlur(pred_mask, (3,3), 0)
    _, pred_mask = cv2.threshold(pred_mask, 127, 255, cv2.THRESH_BINARY)

    # Largest region
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(pred_mask, connectivity=8)

    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        clean_mask = np.zeros_like(pred_mask)
        clean_mask[labels == largest_label] = 255
        pred_mask = clean_mask

    # Final refine
    pred_mask = cv2.dilate(pred_mask, kernel, iterations=1)
    pred_mask = cv2.medianBlur(pred_mask, 3)

    # -----------------------------
    # EXTRACT REGION
    # -----------------------------
    extracted_region = cv2.bitwise_and(brain, brain, mask=pred_mask)

    # -----------------------------
    # SEVERITY CALCULATION
    # -----------------------------
    stroke_pixels = np.sum(pred_mask == 255)
    total_pixels = np.sum(brain > 0)

    stroke_ratio = (stroke_pixels / (total_pixels + 1e-6)) * 100

    # Classification
    if stroke_pixels == 0:
        severity = "Normal"
    elif stroke_ratio < 2:
        severity = "Mild"
    elif stroke_ratio < 5:
        severity = "Moderate"
    else:
        severity = "Severe"

    # -----------------------------
    # OUTPUT
    # -----------------------------
    st.subheader("🎯 Extracted Stroke Region")
    st.image(extracted_region, caption="Pixel-Level Extracted Region", use_container_width=True)

    st.subheader("📊 Severity Analysis")

    st.write(f"**Stroke Area:** {stroke_ratio:.2f}%")

    if severity == "Normal":
        st.success(f"🟢 Severity: {severity}")
    elif severity == "Mild":
        st.info(f"🟡 Severity: {severity}")
    elif severity == "Moderate":
        st.warning(f"🟠 Severity: {severity}")
    else:
        st.error(f"🔴 Severity: {severity}")

    # -----------------------------
    # SAVE
    # -----------------------------
    if st.button("💾 Save Output"):
        cv2.imwrite("stroke_region.png", extracted_region)
        st.success("Saved as stroke_region.png")

Writing app.py


In [ ]:
!pip install localtunnel
!streamlit run app.py & npx localtunnel --port 8501

ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel
⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://silver-falcons-wink.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.115.52.7:8501

  Stopping...
^C
